In [3]:
import pandas as pd
import sqlite3

# 1. Load the raw datafile
file_path = "online_retail_II.csv"
raw_df = pd.read_csv(file_path, encoding='ISO-8859-1')

# 2. Set up a local SQLite in-memory database
conn = sqlite3.connect(':memory:')
raw_df.to_sql('retail_transactions', conn, index=False)

# 3. Clean data and calculate financial baseline using SQL
query = """
    SELECT 
        Invoice, 
        StockCode, 
        Description, 
        Quantity, 
        InvoiceDate, 
        Price, 
        "Customer ID", 
        Country,
        (Quantity * Price) AS Revenue
    FROM retail_transactions
    WHERE "Customer ID" IS NOT NULL 
      AND Quantity > 0
"""

# 4. Pull the cleaned data back into a Pandas DataFrame
df = pd.read_sql_query(query, conn)

# 5. Fix dates for subsequent RFM and Cohort analysis
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 186406 entries, 0 to 186405
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      186406 non-null  str           
 1   StockCode    186406 non-null  str           
 2   Description  186406 non-null  str           
 3   Quantity     186406 non-null  int64         
 4   InvoiceDate  186406 non-null  datetime64[us]
 5   Price        186406 non-null  float64       
 6   Customer ID  186406 non-null  float64       
 7   Country      186405 non-null  str           
 8   Revenue      186406 non-null  float64       
dtypes: datetime64[us](1), float64(3), int64(1), str(4)
memory usage: 12.8 MB
None


In [4]:
# Define a function to categorize customers based on their RFM score
def segment_customer(score):
    if score == '444':
        return 'Champions'
    elif score.startswith('3') or score.startswith('4'):
        return 'Loyal Customers'
    elif score.startswith('1') and (score.endswith('3') or score.endswith('4')):
        return 'At Risk (High Value)'
    elif score.startswith('1') or score.startswith('2'):
        return 'Needs Attention / Lost'
    else:
        return 'Average Customers'

# Apply the function to create a new column
rfm['Customer_Profile'] = rfm['RFM_Segment'].apply(segment_customer)

NameError: name 'rfm' is not defined

In [ ]:
import datetime as dt

# 1. Establish a snapshot date (one day after the latest transaction in th
# et)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# 2. Aggregate metrics per customer
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency: days since last purchase
    'Invoice': 'nunique',                                     # Frequency: total distinct orders
    'Revenue': 'sum'                                          # Monetary: total money spent
}).rename(columns={'InvoiceDate': 'Recency', 
                   'Invoice': 'Frequency', 
                   'Revenue': 'Monetary'})

# 3. Assign quartile scores from 1 (worst) to 4 (best)
# Recency is scored inversely: fewer days since last purchase equals a higher score (4)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1])

# Frequency and Monetary use rank(method='first') to prevent bin edge errors on duplicate values
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4])

# 4. Concatenate scores into a single segment string (e.g., '444' for top customers)
rfm['RFM_Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print(rfm.head())

In [ ]:
# Define a function to categorize customers based on their RFM score
def segment_customer(score):
    if score == '444':
        return 'Champions'
    elif score.startswith('3') or score.startswith('4'):
        return 'Loyal Customers'
    elif score.startswith('1') and (score.endswith('3') or score.endswith('4')):
        return 'At Risk (High Value)'
    elif score.startswith('1') or score.startswith('2'):
        return 'Needs Attention / Lost'
    else:
        return 'Average Customers'

# Apply the function to create a new column
rfm['Customer_Profile'] = rfm['RFM_Segment'].apply(segment_customer)

NameError: name 'rfm' is not defined

In [ ]:
# Extract the month from the InvoiceDate
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

# Find the first month each customer made a purchase (their Cohort)
df['CohortMonth'] = df.groupby('Customer ID')['InvoiceDate'].transform('min').dt.to_period('M')

# Calculate the difference in months between the invoice and the first purchase
df['CohortIndex'] = (df['InvoiceMonth'].dt.year - df['CohortMonth'].dt.year) * 12 + \
                    (df['InvoiceMonth'].dt.month - df['CohortMonth'].dt.month) + 1

# Count unique customers per cohort and month, then pivot the table
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()
retention_matrix = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='Customer ID')

# Convert the raw customer counts into percentage retention rates
cohort_sizes = retention_matrix.iloc[:, 0]
retention_percentages = retention_matrix.divide(cohort_sizes, axis=0).round(3) * 100

print(retention_percentages.head())